# Pharmacy Data Analytics Using Jupyter

This notebook answers 15 pharmacy data analytics questions using `pandas`, `numpy`, `seaborn`, and `matplotlib`.

It uses the dataset in `data/pharmacy_30k/` and saves charts plus CSV outputs to `outputs/data_analytic_using_jupyter/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "seaborn is required to run this notebook. Install it with: pip install seaborn"
    ) from exc

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data' / 'pharmacy_30k'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'data_analytic_using_jupyter'
AS_OF_DATE = pd.Timestamp('2026-04-22')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

In [ ]:
medicines = pd.read_csv(DATA_DIR / 'medicine_master.csv')
inventory = pd.read_csv(DATA_DIR / 'inventory_batches.csv')
sales = pd.read_csv(DATA_DIR / 'sales_transactions.csv')

sales['sale_date'] = pd.to_datetime(sales['sale_date'])
inventory['purchase_date'] = pd.to_datetime(inventory['purchase_date'])
inventory['expiry_date'] = pd.to_datetime(inventory['expiry_date'])

sales['quantity_sold'] = pd.to_numeric(sales['quantity_sold'])
sales['unit_price'] = pd.to_numeric(sales['unit_price'])
sales['discount_pct'] = pd.to_numeric(sales['discount_pct'])
sales['total_amount'] = pd.to_numeric(sales['total_amount'])
inventory['opening_stock'] = pd.to_numeric(inventory['opening_stock'])
inventory['stock_in_hand'] = pd.to_numeric(inventory['stock_in_hand'])
inventory['unit_cost'] = pd.to_numeric(inventory['unit_cost'])
inventory['inventory_value'] = pd.to_numeric(inventory['inventory_value'])

sales_enriched = (
    sales.merge(medicines, on='medicine_id', how='left')
    .merge(
        inventory[['batch_no', 'medicine_id', 'expiry_date', 'stock_in_hand', 'supplier_name', 'unit_cost']],
        on=['batch_no', 'medicine_id'],
        how='left'
    )
)

sales_enriched['month'] = sales_enriched['sale_date'].dt.to_period('M').astype(str)
inventory['days_to_expiry'] = (inventory['expiry_date'] - AS_OF_DATE).dt.days
inventory['expiry_bucket'] = pd.cut(
    inventory['days_to_expiry'],
    bins=[-10000, -1, 30, 60, 90, 10000],
    labels=['Expired', '0-30 days', '31-60 days', '61-90 days', '90+ days']
)

print('Medicines:', len(medicines))
print('Inventory batches:', len(inventory))
print('Sales transactions:', len(sales))

In [ ]:
def save_plot(filename):
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()

def save_table(df, filename):
    df.to_csv(OUTPUT_DIR / filename, index=False)

## Q1. Which medicine categories generate the highest total sales?

In [ ]:
q1 = (
    sales_enriched.groupby('category', as_index=False)['total_amount']
    .sum()
    .rename(columns={'total_amount': 'total_sales'})
    .sort_values('total_sales', ascending=False)
)
save_table(q1, 'q01_category_sales.csv')
sns.barplot(data=q1, x='category', y='total_sales', color='#2a6f97')
plt.title('Q1. Category-Wise Total Sales')
plt.xticks(rotation=30, ha='right')
save_plot('q01_category_sales.png')
q1

## Q2. Which individual medicines are the top-selling by revenue?

In [ ]:
q2 = (
    sales_enriched.groupby(['medicine_id', 'medicine_name'], as_index=False)['total_amount']
    .sum()
    .rename(columns={'total_amount': 'revenue'})
    .sort_values('revenue', ascending=False)
    .head(10)
)
save_table(q2, 'q02_top_medicines_by_revenue.csv')
sns.barplot(data=q2, x='revenue', y='medicine_name', palette='Blues_r')
plt.title('Q2. Top 10 Medicines by Revenue')
save_plot('q02_top_medicines_by_revenue.png')
q2

## Q3. Which medicines sell the most by quantity?

In [ ]:
q3 = (
    sales_enriched.groupby(['medicine_id', 'medicine_name'], as_index=False)['quantity_sold']
    .sum()
    .sort_values('quantity_sold', ascending=False)
    .head(10)
)
save_table(q3, 'q03_top_medicines_by_quantity.csv')
sns.barplot(data=q3, x='quantity_sold', y='medicine_name', palette='Greens_r')
plt.title('Q3. Top 10 Medicines by Quantity Sold')
save_plot('q03_top_medicines_by_quantity.png')
q3

## Q4. What is the monthly sales trend over time?

In [ ]:
q4 = (
    sales_enriched.groupby('month', as_index=False)['total_amount']
    .sum()
    .rename(columns={'total_amount': 'monthly_sales'})
)
save_table(q4, 'q04_monthly_sales_trend.csv')
sns.lineplot(data=q4, x='month', y='monthly_sales', marker='o', color='#1d3557')
plt.title('Q4. Monthly Sales Trend')
plt.xticks(rotation=30, ha='right')
save_plot('q04_monthly_sales_trend.png')
q4

## Q5. Which categories are growing and which are declining?

In [ ]:
monthly_category = (
    sales_enriched.groupby(['month', 'category'], as_index=False)['total_amount']
    .sum()
    .rename(columns={'total_amount': 'monthly_sales'})
)
pivot = monthly_category.pivot(index='month', columns='category', values='monthly_sales').fillna(0)
growth = ((pivot.iloc[-1] - pivot.iloc[0]) / pivot.iloc[0].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)
q5 = growth.reset_index()
q5.columns = ['category', 'growth_rate']
q5 = q5.sort_values('growth_rate', ascending=False)
save_table(q5, 'q05_category_growth_decline.csv')
sns.barplot(data=q5, x='growth_rate', y='category', palette='coolwarm')
plt.title('Q5. Category Growth vs Decline')
save_plot('q05_category_growth_decline.png')
q5

## Q6. Which medicines are slow-moving and may be overstocked?

In [ ]:
q6 = (
    sales_enriched.groupby(['medicine_id', 'medicine_name', 'category'], as_index=False)['quantity_sold']
    .sum()
    .assign(avg_daily_units=lambda df: df['quantity_sold'] / 365)
    .sort_values('avg_daily_units', ascending=True)
    .head(10)
)
save_table(q6, 'q06_slow_moving_medicines.csv')
sns.barplot(data=q6, x='avg_daily_units', y='medicine_name', palette='Reds')
plt.title('Q6. Slow-Moving Medicines')
save_plot('q06_slow_moving_medicines.png')
q6

## Q7. Which batches are already expired but still have stock remaining?

In [ ]:
q7 = (
    inventory.loc[(inventory['days_to_expiry'] < 0) & (inventory['stock_in_hand'] > 0)]
    .merge(medicines[['medicine_id', 'medicine_name', 'category']], on='medicine_id', how='left')
    .sort_values('stock_in_hand', ascending=False)
)
save_table(q7, 'q07_expired_batches_with_stock.csv')
plot_q7 = q7.head(10)
sns.barplot(data=plot_q7, x='stock_in_hand', y='batch_no', palette='rocket')
plt.title('Q7. Expired Batches Still Holding Stock')
save_plot('q07_expired_batches_with_stock.png')
q7.head(10)

## Q8. Which batches will expire in the next 30, 60, or 90 days?

In [ ]:
q8 = (
    inventory.loc[inventory['expiry_bucket'].isin(['0-30 days', '31-60 days', '61-90 days'])]
    .groupby('expiry_bucket', as_index=False)['batch_no']
    .count()
    .rename(columns={'batch_no': 'batch_count'})
)
save_table(q8, 'q08_near_expiry_buckets.csv')
sns.barplot(data=q8, x='expiry_bucket', y='batch_count', palette='magma')
plt.title('Q8. Batches Expiring in 30, 60, 90 Days')
save_plot('q08_near_expiry_buckets.png')
q8

## Q9. What is the total inventory value at expiry risk?

In [ ]:
q9 = (
    inventory.assign(risk_value=inventory['stock_in_hand'] * inventory['unit_cost'])
    .groupby('expiry_bucket', as_index=False)['risk_value']
    .sum()
)
save_table(q9, 'q09_inventory_value_at_risk.csv')
sns.barplot(data=q9, x='expiry_bucket', y='risk_value', palette='flare')
plt.title('Q9. Inventory Value at Expiry Risk')
save_plot('q09_inventory_value_at_risk.png')
q9

## Q10. Which medicines have high stock but low sales velocity?

In [ ]:
velocity_q10 = (
    sales_enriched.groupby('medicine_id', as_index=False)['quantity_sold']
    .sum()
    .assign(avg_daily_units=lambda df: df['quantity_sold'] / 365)
)
stock_q10 = inventory.groupby('medicine_id', as_index=False)['stock_in_hand'].sum()
q10 = (
    stock_q10.merge(velocity_q10, on='medicine_id', how='left')
    .merge(medicines[['medicine_id', 'medicine_name', 'category']], on='medicine_id', how='left')
    .fillna({'avg_daily_units': 0})
    .assign(stock_to_velocity_ratio=lambda df: np.where(df['avg_daily_units'] > 0, df['stock_in_hand'] / df['avg_daily_units'], np.nan))
    .sort_values(['stock_in_hand', 'avg_daily_units'], ascending=[False, True])
    .head(15)
)
save_table(q10, 'q10_high_stock_low_velocity.csv')
sns.scatterplot(data=q10, x='avg_daily_units', y='stock_in_hand', hue='category', s=120)
plt.title('Q10. High Stock but Low Sales Velocity')
save_plot('q10_high_stock_low_velocity.png')
q10

## Q11. Which products should be prioritized for discount or clearance before expiry?

In [ ]:
velocity_q11 = (
    sales_enriched.groupby('medicine_id', as_index=False)['quantity_sold']
    .sum()
    .assign(avg_daily_units=lambda df: df['quantity_sold'] / 365)
)
q11 = (
    inventory.merge(velocity_q11, on='medicine_id', how='left')
    .merge(medicines[['medicine_id', 'medicine_name', 'category']], on='medicine_id', how='left')
    .fillna({'avg_daily_units': 0})
    .assign(projected_units_before_expiry=lambda df: np.maximum(df['days_to_expiry'], 0) * df['avg_daily_units'])
    .assign(estimated_units_at_risk=lambda df: np.maximum(df['stock_in_hand'] - df['projected_units_before_expiry'], 0))
    .loc[lambda df: df['estimated_units_at_risk'] > 0]
    .sort_values('estimated_units_at_risk', ascending=False)
    .head(10)
)
save_table(q11, 'q11_discount_clearance_candidates.csv')
sns.barplot(data=q11, x='estimated_units_at_risk', y='batch_no', palette='OrRd')
plt.title('Q11. Discount or Clearance Candidates')
save_plot('q11_discount_clearance_candidates.png')
q11

## Q12. Which products need urgent reorder because sales are high and stock is low?

In [ ]:
velocity_q12 = (
    sales_enriched.groupby('medicine_id', as_index=False)['quantity_sold']
    .sum()
    .assign(avg_daily_units=lambda df: df['quantity_sold'] / 365)
)
stock_q12 = inventory.groupby('medicine_id', as_index=False)['stock_in_hand'].sum()
q12 = (
    velocity_q12.merge(stock_q12, on='medicine_id', how='left')
    .merge(medicines[['medicine_id', 'medicine_name', 'category']], on='medicine_id', how='left')
    .assign(days_of_cover=lambda df: np.where(df['avg_daily_units'] > 0, df['stock_in_hand'] / df['avg_daily_units'], np.nan))
    .sort_values(['avg_daily_units', 'days_of_cover'], ascending=[False, True])
    .head(10)
)
save_table(q12, 'q12_urgent_reorder_candidates.csv')
sns.barplot(data=q12, x='days_of_cover', y='medicine_name', palette='crest')
plt.title('Q12. Urgent Reorder Candidates')
save_plot('q12_urgent_reorder_candidates.png')
q12

## Q13. Which suppliers are linked to high-risk or low-rotation inventory?

In [ ]:
q13 = (
    inventory.assign(risk_value=inventory['stock_in_hand'] * inventory['unit_cost'])
    .groupby('supplier_name', as_index=False)['risk_value']
    .sum()
    .sort_values('risk_value', ascending=False)
)
save_table(q13, 'q13_supplier_risk.csv')
sns.barplot(data=q13, x='risk_value', y='supplier_name', palette='viridis')
plt.title('Q13. Supplier-Linked Inventory Risk')
save_plot('q13_supplier_risk.png')
q13

## Q14. Which categories contribute most to potential expiry loss?

In [ ]:
q14 = (
    inventory.merge(medicines[['medicine_id', 'category']], on='medicine_id', how='left')
    .assign(risk_value=lambda df: df['stock_in_hand'] * df['unit_cost'])
    .groupby('category', as_index=False)['risk_value']
    .sum()
    .sort_values('risk_value', ascending=False)
)
save_table(q14, 'q14_category_expiry_loss.csv')
sns.barplot(data=q14, x='risk_value', y='category', palette='cubehelix')
plt.title('Q14. Category Contribution to Expiry Risk')
save_plot('q14_category_expiry_loss.png')
q14

## Q15. What actions should the pharmacy take to reduce expiry loss while protecting sales?

In [ ]:
velocity_q15 = (
    sales_enriched.groupby('medicine_id', as_index=False)['quantity_sold']
    .sum()
    .assign(avg_daily_units=lambda df: df['quantity_sold'] / 365)
)
stock_q15 = inventory.groupby('medicine_id', as_index=False)['stock_in_hand'].sum()
nearest_expiry_q15 = inventory.groupby('medicine_id', as_index=False)['days_to_expiry'].min()

q15_detail = (
    medicines[['medicine_id', 'medicine_name', 'category']]
    .merge(stock_q15, on='medicine_id', how='left')
    .merge(velocity_q15[['medicine_id', 'avg_daily_units']], on='medicine_id', how='left')
    .merge(nearest_expiry_q15, on='medicine_id', how='left')
    .fillna({'stock_in_hand': 0, 'avg_daily_units': 0, 'days_to_expiry': 9999})
)

conditions = [
    (q15_detail['days_to_expiry'] <= 30) & (q15_detail['stock_in_hand'] > q15_detail['avg_daily_units'] * 20),
    (q15_detail['avg_daily_units'] >= q15_detail['avg_daily_units'].quantile(0.75)) & (q15_detail['stock_in_hand'] < q15_detail['avg_daily_units'] * 15),
    (q15_detail['stock_in_hand'] > q15_detail['stock_in_hand'].quantile(0.75)) & (q15_detail['avg_daily_units'] < q15_detail['avg_daily_units'].quantile(0.25)),
]
choices = ['Discount/Clear', 'Reorder', 'Monitor Overstock']
q15_detail['recommended_action'] = np.select(conditions, choices, default='Maintain')

q15_summary = (
    q15_detail.groupby('recommended_action', as_index=False)['medicine_id']
    .count()
    .rename(columns={'medicine_id': 'medicine_count'})
    .sort_values('medicine_count', ascending=False)
)

save_table(q15_detail.sort_values(['recommended_action', 'medicine_name']), 'q15_action_recommendations_detail.csv')
save_table(q15_summary, 'q15_action_recommendations_summary.csv')
sns.barplot(data=q15_summary, x='recommended_action', y='medicine_count', palette='Set2')
plt.title('Q15. Recommended Pharmacy Actions')
plt.xticks(rotation=20, ha='right')
save_plot('q15_action_recommendations.png')
q15_summary

## Output Location

All generated CSV tables and PNG charts are saved in:

`outputs/data_analytic_using_jupyter/`